<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----


### Вариант задания 



<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;
using System.Collections.Generic;

public delegate void StockChangedHandler(Product product, int oldQuantity, int newQuantity);
public delegate double PriceModifier(double price);

public class PriceChangedEventArgs : EventArgs
{
    public double OldPrice { get; private set; }
    public double NewPrice { get; private set; }

    public PriceChangedEventArgs(double oldPrice, double newPrice)
    {
        OldPrice = oldPrice;
        NewPrice = newPrice;
    }
}

public interface ILogger
{
    void Log(string message);
}

public class ConsoleLogger : ILogger
{
    public void Log(string message)
    {
        Console.WriteLine(message);
    }
}

public class PrefixLogger : ILogger
{
    private string _prefix;

    public PrefixLogger(string prefix)
    {
        _prefix = prefix;
    }

    public void Log(string message)
    {
        Console.WriteLine($"{_prefix} {message}");
    }
}

public class SilentLogger : ILogger
{
    public void Log(string message)
    {
    }
}

public interface IDiscountPolicy
{
    double Apply(double price, double percent);
}

public class StandardDiscountPolicy : IDiscountPolicy
{
    public double Apply(double price, double percent)
    {
        return price - price * percent / 100;
    }
}

public class LimitedDiscountPolicy : IDiscountPolicy
{
    private double _maxPercent;

    public LimitedDiscountPolicy(double maxPercent)
    {
        _maxPercent = maxPercent;
    }

    public double Apply(double price, double percent)
    {
        double real = Math.Min(percent, _maxPercent);
        return price - price * real / 100;
    }
}

public interface INotifier
{
    void Notify(string message);
}

public class ConsoleNotifier : INotifier
{
    public void Notify(string message)
    {
        Console.WriteLine($"[УВЕДОМЛЕНИЕ] {message}");
    }
}

public interface IProductServices
{
    ILogger Logger { get; }
    IDiscountPolicy DiscountPolicy { get; }
}

public class ProductServices : IProductServices
{
    public ILogger Logger { get; private set; }
    public IDiscountPolicy DiscountPolicy { get; private set; }

    public ProductServices(ILogger logger, IDiscountPolicy discountPolicy)
    {
        Logger = logger ?? throw new ArgumentNullException(nameof(logger));
        DiscountPolicy = discountPolicy ?? throw new ArgumentNullException(nameof(discountPolicy));
    }
}

public class ServiceContainer
{
    private readonly Dictionary<Type, Func<ServiceContainer, object>> _map = new Dictionary<Type, Func<ServiceContainer, object>>();

    public void RegisterSingleton<T>(T instance) where T : class
    {
        _map[typeof(T)] = c => instance;
    }

    public void Register<T>(Func<ServiceContainer, T> factory) where T : class
    {
        _map[typeof(T)] = c => factory(c);
    }

    public T Resolve<T>() where T : class
    {
        if (!_map.TryGetValue(typeof(T), out Func<ServiceContainer, object> factory))
            throw new InvalidOperationException($"Сервис {typeof(T).Name} не зарегистрирован!");
        return (T)factory(this);
    }
}

public class OrderService
{
    private ILogger _logger;
    private INotifier _notifier;
    private int _lowStockLimit;
    private List<string> _orderHistory = new List<string>();

    public event Action<Product, int> OrderPlaced;

    public IReadOnlyList<string> OrderHistory
    {
        get { return _orderHistory; }
    }

    public int LowStockLimit
    {
        get { return _lowStockLimit; }
        set { _lowStockLimit = value; }
    }

    public OrderService(ILogger logger, INotifier notifier, int lowStockLimit = 5)
    {
        _logger = logger;
        _notifier = notifier;
        _lowStockLimit = lowStockLimit;
    }

    public bool PlaceOrder(Product product, int count)
    {
        _logger.Log($"Оформление заказа: «{product.Name}» x{count}");
        bool ok = product.Sell(count);
        if (ok)
        {
            _orderHistory.Add($"{product.Name} x{count}");
            OrderPlaced?.Invoke(product, count);
            if (product.Quantity <= _lowStockLimit)
                _notifier.Notify($"Заканчивается товар «{product.Name}»: осталось {product.Quantity} шт.");
        }
        return ok;
    }
}

public class StockMonitor
{
    private INotifier _notifier;
    private Func<double, double, bool> _priceAlertRule;
    private List<string> _alerts = new List<string>();

    public IReadOnlyList<string> Alerts
    {
        get { return _alerts; }
    }

    public StockMonitor(INotifier notifier, Func<double, double, bool> priceAlertRule)
    {
        _notifier = notifier ?? throw new ArgumentNullException(nameof(notifier));
        _priceAlertRule = priceAlertRule ?? throw new ArgumentNullException(nameof(priceAlertRule));
    }

    public void Watch(Product product)
    {
        product.OutOfStock += OnOutOfStock;
        product.PriceChanged += OnPriceChanged;
    }

    public void Unwatch(Product product)
    {
        product.OutOfStock -= OnOutOfStock;
        product.PriceChanged -= OnPriceChanged;
    }

    private void Report(string message)
    {
        _alerts.Add(message);
        _notifier.Notify(message);
    }

    private void OnOutOfStock(Product product)
    {
        Report($"Товар «{product.Name}» закончился на складе");
    }

    private void OnPriceChanged(object sender, PriceChangedEventArgs e)
    {
        Product product = (Product)sender;
        if (_priceAlertRule(e.OldPrice, e.NewPrice))
            Report($"Резкое снижение цены «{product.Name}»: {e.OldPrice} -> {e.NewPrice} руб.");
    }
}

public interface IRechargeable
{
    int BatteryLevel { get; set; }
    void Charge(int percent);
    string GetStatus();
}

public interface IWaterproof
{
    int WaterResistanceMeters { get; set; }
    bool CanSwim();
    string GetStatus();
}

public interface IPlayable
{
    int DurationMinutes { get; set; }
    void Play();
    void Pause();
}

public class Product
{
    private string _name;
    private double _price;
    private string _manufacturer;
    private string _article;
    private int _quantity;
    private string _country;
    private double _rating;
    private string _barcode;
    private int _soldCount;
    private HashSet<string> _tags = new HashSet<string>();
    private List<double> _priceHistory = new List<double>();
    private Dictionary<string, string> _reviews = new Dictionary<string, string>();
    private IProductServices _services;

    public event EventHandler<PriceChangedEventArgs> PriceChanged;
    public event StockChangedHandler StockChanged;
    public event Action<Product> OutOfStock;

    public string Name
    {
        get { return _name; }
        set { _name = value; }
    }

    public double Price
    {
        get { return _price; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Цена не может быть отрицательной!");
            double oldPrice = _price;
            _price = value;
            _priceHistory.Add(value);
            if (oldPrice != value)
                OnPriceChanged(new PriceChangedEventArgs(oldPrice, value));
        }
    }

    public string Manufacturer
    {
        get { return _manufacturer; }
        set { _manufacturer = value; }
    }

    public string Article
    {
        get { return _article; }
        set { _article = value; }
    }

    public int Quantity
    {
        get { return _quantity; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Количество не может быть отрицательным!");
            int oldQuantity = _quantity;
            _quantity = value;
            if (oldQuantity != value)
                OnStockChanged(oldQuantity, value);
        }
    }

    public string Country
    {
        get { return _country; }
        set { _country = value; }
    }

    public double Rating
    {
        get { return _rating; }
        private set { _rating = value; }
    }

    public string Barcode
    {
        get { return _barcode; }
        set { _barcode = value; }
    }

    public int SoldCount
    {
        get { return _soldCount; }
        private set { _soldCount = value; }
    }

    public IReadOnlyCollection<string> Tags
    {
        get { return _tags; }
    }

    public IReadOnlyList<double> PriceHistory
    {
        get { return _priceHistory; }
    }

    public IReadOnlyDictionary<string, string> Reviews
    {
        get { return _reviews; }
    }

    protected ILogger Logger
    {
        get { return _services.Logger; }
    }

    protected IDiscountPolicy DiscountPolicy
    {
        get { return _services.DiscountPolicy; }
    }

    public Product(string name, double price, string manufacturer, string article, int quantity, string country,
                   IProductServices services)
    {
        _services = services ?? throw new ArgumentNullException(nameof(services));
        Name = name;
        Price = price;
        Manufacturer = manufacturer;
        Article = article;
        Quantity = quantity;
        Country = country;
        Rating = 0;
        Barcode = "000000";
        SoldCount = 0;
    }

    protected virtual void OnPriceChanged(PriceChangedEventArgs e)
    {
        PriceChanged?.Invoke(this, e);
    }

    protected virtual void OnStockChanged(int oldQuantity, int newQuantity)
    {
        StockChanged?.Invoke(this, oldQuantity, newQuantity);
        if (newQuantity == 0 && oldQuantity > 0)
            OutOfStock?.Invoke(this);
    }

    public virtual string GetCategory()
    {
        return "Товар";
    }

    public virtual string GetInfo()
    {
        return $"Название: {Name}, Цена: {Price} руб., Производитель: {Manufacturer}";
    }

    public virtual void Display()
    {
        Console.WriteLine(GetInfo());
    }

    public virtual void Discount(double percent)
    {
        Price = DiscountPolicy.Apply(Price, percent);
        Logger.Log($"{Name}: скидка {percent}%. Новая цена: {Price} руб.");
    }

    public void Discount(double percent, double maxAmount)
    {
        double amount = Math.Min(Price * percent / 100, maxAmount);
        Price -= amount;
        Logger.Log($"{Name}: скидка {amount} руб. (не более {maxAmount} руб.). Новая цена: {Price} руб.");
    }

    public void AddToStock(int count)
    {
        if (count <= 0)
            throw new ArgumentOutOfRangeException(nameof(count), "Количество должно быть больше нуля!");
        Quantity += count;
        Logger.Log($"Склад пополнен: «{Name}» +{count} шт. Всего: {Quantity} шт.");
    }

    public void AddToStock(int count, string supplier)
    {
        AddToStock(count);
        Logger.Log($"Поставщик: {supplier}");
    }

    public bool Sell(int count)
    {
        if (count <= 0)
            throw new ArgumentOutOfRangeException(nameof(count), "Количество должно быть больше нуля!");
        if (count > Quantity)
        {
            Logger.Log($"Недостаточно товара «{Name}»: на складе {Quantity} шт., запрошено {count} шт.");
            return false;
        }
        Quantity -= count;
        SoldCount += count;
        Logger.Log($"Продано: «{Name}» {count} шт. на сумму {Price * count} руб. Остаток: {Quantity} шт.");
        return true;
    }

    public bool Sell(int count, double discountPercent)
    {
        double oldPrice = _price;
        _price = DiscountPolicy.Apply(oldPrice, discountPercent);
        bool result = Sell(count);
        _price = oldPrice;
        return result;
    }

    public double GetTotalCost()
    {
        return Price * Quantity;
    }

    public bool IsInStock()
    {
        return Quantity > 0;
    }

    public void Rate(double value)
    {
        if (value < 0 || value > 5)
            throw new ArgumentOutOfRangeException(nameof(value), "Оценка должна быть от 0 до 5!");
        Rating = Rating == 0 ? value : Math.Round((Rating + value) / 2, 1);
        Logger.Log($"«{Name}»: рейтинг {Rating}");
    }

    public double GetPriceWithVat(double vatPercent = 20)
    {
        return Price * (1 + vatPercent / 100);
    }

    public string GetStockStatus()
    {
        if (Quantity == 0) return "Нет в наличии";
        if (Quantity < 5) return "Заканчивается";
        return "В наличии";
    }

    public void ComparePriceWith(Product otherProduct)
    {
        if (Price > otherProduct.Price)
            Logger.Log($"Товар «{Name}» дороже, чем «{otherProduct.Name}» на {Price - otherProduct.Price} руб.");
        else if (Price < otherProduct.Price)
            Logger.Log($"Товар «{Name}» дешевле, чем «{otherProduct.Name}» на {otherProduct.Price - Price} руб.");
        else
            Logger.Log($"Товары «{Name}» и «{otherProduct.Name}» стоят одинаково.");
    }

    public void ComparePriceWith(double price)
    {
        if (Price > price)
            Logger.Log($"Товар «{Name}» дороже {price} руб. на {Price - price} руб.");
        else if (Price < price)
            Logger.Log($"Товар «{Name}» дешевле {price} руб. на {price - Price} руб.");
        else
            Logger.Log($"Товар «{Name}» стоит ровно {price} руб.");
    }

    public void ApplyBulkDiscount(Product secondProduct, double bundleDiscountPercent)
    {
        Logger.Log($"[Комплект «{Name}» + «{secondProduct.Name}»]");
        Discount(bundleDiscountPercent);
        secondProduct.Discount(bundleDiscountPercent);
    }

    public bool AddTag(string tag)
    {
        if (string.IsNullOrWhiteSpace(tag))
            throw new ArgumentException("Тег не может быть пустым!", nameof(tag));
        bool added = _tags.Add(tag.ToLower());
        if (added)
            Logger.Log($"«{Name}»: добавлен тег «{tag}»");
        return added;
    }

    public bool HasTag(string tag)
    {
        return _tags.Contains(tag.ToLower());
    }

    public void AddReview(string author, string text)
    {
        bool isNew = !_reviews.ContainsKey(author);
        _reviews[author] = text;
        Logger.Log($"«{Name}»: {(isNew ? "новый отзыв" : "отзыв обновлён")} от {author}");
    }

    public void ApplyPriceModifier(PriceModifier modifier)
    {
        if (modifier == null)
            throw new ArgumentNullException(nameof(modifier));
        double oldPrice = Price;
        Price = modifier(oldPrice);
        Logger.Log($"«{Name}»: цена изменена {oldPrice} -> {Price} руб.");
    }

    public double GetLowestPrice()
    {
        double lowest = _priceHistory[0];
        foreach (double value in _priceHistory)
        {
            if (value < lowest)
                lowest = value;
        }
        return lowest;
    }
}

public class Electronics : Product
{
    private int _warrantyPeriod;
    private string _model;
    private int _powerWatts;
    private string _color;
    private string _energyClass;
    private int _voltage;
    private List<string> _accessories = new List<string>();
    private List<string> _repairLog = new List<string>();
    private bool _isPoweredOn;

    public event Action<Electronics, bool> PowerStateChanged;

    public int WarrantyPeriod
    {
        get { return _warrantyPeriod; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Гарантия не может быть отрицательной!");
            _warrantyPeriod = value;
        }
    }

    public string Model
    {
        get { return _model; }
        set { _model = value; }
    }

    public int PowerWatts
    {
        get { return _powerWatts; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Мощность не может быть отрицательной!");
            _powerWatts = value;
        }
    }

    public string Color
    {
        get { return _color; }
        set { _color = value; }
    }

    public string EnergyClass
    {
        get { return _energyClass; }
        set { _energyClass = value; }
    }

    public int Voltage
    {
        get { return _voltage; }
        set { _voltage = value; }
    }

    public IReadOnlyList<string> Accessories
    {
        get { return _accessories; }
    }

    public IReadOnlyList<string> RepairLog
    {
        get { return _repairLog; }
    }

    public bool IsPoweredOn
    {
        get { return _isPoweredOn; }
        private set { _isPoweredOn = value; }
    }

    public Electronics(string name, double price, string manufacturer, string article, int quantity, string country,
                       int warrantyPeriod, string model, int powerWatts, string color, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, services)
    {
        WarrantyPeriod = warrantyPeriod;
        Model = model;
        PowerWatts = powerWatts;
        Color = color;
        EnergyClass = "A";
        Voltage = 220;
        IsPoweredOn = false;
    }

    public override string GetCategory()
    {
        return "Электроника";
    }

    public override void Display()
    {
        Console.WriteLine(GetInfo());
        Console.WriteLine(GetTechInfo());
    }

    public override void Discount(double percent)
    {
        double extra = WarrantyPeriod > 12 ? 5 : 0;
        base.Discount(percent + extra);
    }

    public bool IsUnderWarranty(int monthsUsed)
    {
        return monthsUsed <= WarrantyPeriod;
    }

    public double CalculateEnergyCost(double hours, double tariffPerKWh)
    {
        return PowerWatts / 1000.0 * hours * tariffPerKWh;
    }

    public void ExtendWarranty(int months)
    {
        WarrantyPeriod += months;
        Logger.Log($"Гарантия «{Name}» продлена на {months} мес. Теперь: {WarrantyPeriod} мес.");
    }

    public bool IsEnergyEfficient()
    {
        return EnergyClass == "A" || EnergyClass == "A+" || EnergyClass == "A++";
    }

    public virtual string GetTechInfo()
    {
        return $"Модель: {Model}, Мощность: {PowerWatts} Вт, Цвет: {Color}, Гарантия: {WarrantyPeriod} мес.";
    }

    public void AddAccessory(string accessory)
    {
        _accessories.Add(accessory);
        Logger.Log($"«{Name}»: добавлен аксессуар «{accessory}»");
    }

    public void TogglePower()
    {
        IsPoweredOn = !IsPoweredOn;
        Logger.Log($"«{Name}»: питание {(IsPoweredOn ? "включено" : "выключено")}");
        PowerStateChanged?.Invoke(this, IsPoweredOn);
    }

    public void RegisterRepair(string description)
    {
        _repairLog.Add(description);
        Logger.Log($"«{Name}»: ремонт зарегистрирован — {description}");
    }

    public List<string> FindAccessories(Func<string, bool> filter)
    {
        List<string> result = new List<string>();
        foreach (string accessory in _accessories)
        {
            if (filter(accessory))
                result.Add(accessory);
        }
        return result;
    }
}

public class Clothing : Product
{
    private string _size;
    private string _material;
    private string _season;
    private string _color;
    private string _gender;
    private bool _isOnSale;
    private Dictionary<string, int> _sizeStock = new Dictionary<string, int>();
    private HashSet<string> _careTags = new HashSet<string>();
    private int _washCount;

    public event Action<Clothing, string> SizeSoldOut;

    public string Size
    {
        get { return _size; }
        set { _size = value; }
    }

    public string Material
    {
        get { return _material; }
        set { _material = value; }
    }

    public string Season
    {
        get { return _season; }
        set { _season = value; }
    }

    public string Color
    {
        get { return _color; }
        set { _color = value; }
    }

    public string Gender
    {
        get { return _gender; }
        set { _gender = value; }
    }

    public bool IsOnSale
    {
        get { return _isOnSale; }
        private set { _isOnSale = value; }
    }

    public IReadOnlyDictionary<string, int> SizeStock
    {
        get { return _sizeStock; }
    }

    public IReadOnlyCollection<string> CareTags
    {
        get { return _careTags; }
    }

    public int WashCount
    {
        get { return _washCount; }
        private set { _washCount = value; }
    }

    public Clothing(string name, double price, string manufacturer, string article, int quantity, string country,
                    string size, string material, string season, string color, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, services)
    {
        Size = size;
        Material = material;
        Season = season;
        Color = color;
        Gender = "Унисекс";
        IsOnSale = false;
        WashCount = 0;
    }

    public override string GetCategory()
    {
        return "Одежда";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Размер: {Size}, Материал: {Material}";
    }

    public bool IsSuitableForSeason(string season)
    {
        return Season.Equals(season, StringComparison.OrdinalIgnoreCase) || Season.Equals("Всесезонная", StringComparison.OrdinalIgnoreCase);
    }

    public string GetCareInstructions()
    {
        switch (Material.ToLower())
        {
            case "хлопок": return "Стирка при 40°C, можно гладить.";
            case "шерсть": return "Только ручная стирка при 30°C.";
            case "синтетика": return "Деликатная стирка, не гладить горячим утюгом.";
            default: return "Следуйте информации на ярлыке.";
        }
    }

    public void ChangeSize(string newSize)
    {
        Logger.Log($"Размер «{Name}» изменён: {Size} -> {newSize}");
        Size = newSize;
    }

    public void PutOnSale(double percent)
    {
        Discount(percent);
        IsOnSale = true;
    }

    public string GetSizeInfo()
    {
        return $"{Name}: размер {Size}, предназначено для: {Gender}";
    }

    private int GetSizeStockCount(string size)
    {
        int count;
        _sizeStock.TryGetValue(size, out count);
        return count;
    }

    public void AddSizeStock(string size, int count)
    {
        if (count <= 0)
            throw new ArgumentOutOfRangeException(nameof(count), "Количество должно быть больше нуля!");
        _sizeStock[size] = GetSizeStockCount(size) + count;
        Logger.Log($"«{Name}»: размер {size} — в наличии {_sizeStock[size]} шт.");
    }

    public bool SellSize(string size, int count)
    {
        int available = GetSizeStockCount(size);
        if (available < count)
        {
            Logger.Log($"«{Name}»: размера {size} недостаточно (есть {available} шт., нужно {count} шт.).");
            return false;
        }
        if (!Sell(count))
            return false;
        _sizeStock[size] = available - count;
        if (_sizeStock[size] == 0)
            SizeSoldOut?.Invoke(this, size);
        return true;
    }

    public void RecordWash()
    {
        WashCount++;
        Logger.Log($"«{Name}»: количество стирок — {WashCount}.");
    }

    public bool AddCareTag(string tag)
    {
        return _careTags.Add(tag);
    }
}

public class Books : Product
{
    private string _author;
    private string _genre;
    private int _pages;
    private int _year;
    private string _publisher;
    private string _isbn;
    private List<string> _chapters = new List<string>();
    private Dictionary<string, int> _bookmarks = new Dictionary<string, int>();
    private int _readPages;

    public event Action<Books> BookFinished;

    public string Author
    {
        get { return _author; }
        set { _author = value; }
    }

    public string Genre
    {
        get { return _genre; }
        set { _genre = value; }
    }

    public int Pages
    {
        get { return _pages; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Количество страниц должно быть положительным!");
            _pages = value;
        }
    }

    public int Year
    {
        get { return _year; }
        set { _year = value; }
    }

    public string Publisher
    {
        get { return _publisher; }
        set { _publisher = value; }
    }

    public string Isbn
    {
        get { return _isbn; }
        set { _isbn = value; }
    }

    public IReadOnlyList<string> Chapters
    {
        get { return _chapters; }
    }

    public IReadOnlyDictionary<string, int> Bookmarks
    {
        get { return _bookmarks; }
    }

    public int ReadPages
    {
        get { return _readPages; }
        private set { _readPages = value; }
    }

    public Books(string name, double price, string manufacturer, string article, int quantity, string country,
                 string author, string genre, int pages, int year, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, services)
    {
        Author = author;
        Genre = genre;
        Pages = pages;
        Year = year;
        Publisher = manufacturer;
        Isbn = "не указан";
        ReadPages = 0;
    }

    public override string GetCategory()
    {
        return "Книги";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Автор: {Author}, Жанр: {Genre}";
    }

    public double GetReadingTime(int pagesPerHour = 30)
    {
        return (double)Pages / pagesPerHour;
    }

    public bool IsClassic()
    {
        return DateTime.Now.Year - Year > 100;
    }

    public string GetBookCard()
    {
        return $"«{Name}» — {Author}, жанр: {Genre}, {Pages} стр., {Year} г.";
    }

    public bool IsLongBook()
    {
        return Pages > 500;
    }

    public string GetCitation()
    {
        return $"{Author}. {Name}. — {Publisher}, {Year}. — {Pages} с. ISBN: {Isbn}";
    }

    public void AddChapter(string title)
    {
        _chapters.Add(title);
        Logger.Log($"«{Name}»: добавлена глава «{title}»");
    }

    public void AddBookmark(string bookmarkName, int page)
    {
        if (page < 1 || page > Pages)
            throw new ArgumentOutOfRangeException(nameof(page), "Страница вне диапазона книги!");
        _bookmarks[bookmarkName] = page;
        Logger.Log($"«{Name}»: закладка «{bookmarkName}» на стр. {page}");
    }

    public void ReadTo(int page)
    {
        if (page < 0)
            throw new ArgumentOutOfRangeException(nameof(page), "Страница не может быть отрицательной!");
        int previous = ReadPages;
        ReadPages = Math.Max(previous, Math.Min(page, Pages));
        Logger.Log($"«{Name}»: прочитано {ReadPages} из {Pages} стр.");
        if (previous < Pages && ReadPages == Pages)
            BookFinished?.Invoke(this);
    }

    public double GetReadProgress()
    {
        return (double)ReadPages * 100 / Pages;
    }
}

public class Smartphone : Electronics
{
    private int _storageGb;
    private double _screenInches;
    private string _operatingSystem;
    private int _batteryMah;
    private bool _hasNfc;
    private Dictionary<string, string> _contacts = new Dictionary<string, string>();
    private HashSet<string> _installedApps = new HashSet<string>();
    private Queue<string> _callHistory = new Queue<string>();

    public event Action<Smartphone, string> AppInstalled;

    public int StorageGb
    {
        get { return _storageGb; }
        set { _storageGb = value; }
    }

    public double ScreenInches
    {
        get { return _screenInches; }
        set { _screenInches = value; }
    }

    public string OperatingSystem
    {
        get { return _operatingSystem; }
        set { _operatingSystem = value; }
    }

    public int BatteryMah
    {
        get { return _batteryMah; }
        set { _batteryMah = value; }
    }

    public bool HasNfc
    {
        get { return _hasNfc; }
        set { _hasNfc = value; }
    }

    public IReadOnlyDictionary<string, string> Contacts
    {
        get { return _contacts; }
    }

    public IReadOnlyCollection<string> InstalledApps
    {
        get { return _installedApps; }
    }

    public int CallCount
    {
        get { return _callHistory.Count; }
    }

    public Smartphone(string name, double price, string manufacturer, string article, int quantity, string country,
                      int warrantyPeriod, string model, int powerWatts, string color,
                      int storageGb, double screenInches, string operatingSystem, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, warrantyPeriod, model, powerWatts, color, services)
    {
        StorageGb = storageGb;
        ScreenInches = screenInches;
        OperatingSystem = operatingSystem;
        BatteryMah = 4500;
        HasNfc = true;
    }

    public override string GetCategory()
    {
        return "Электроника / Смартфоны";
    }

    public override string GetTechInfo()
    {
        return base.GetTechInfo() + $", Память: {StorageGb} ГБ, Экран: {ScreenInches}\", ОС: {OperatingSystem}";
    }

    public void MakeCall(string number)
    {
        _callHistory.Enqueue(number);
        if (_callHistory.Count > 10)
            _callHistory.Dequeue();
        Logger.Log($"{Name} ({Model}): звонок на номер {number}...");
    }

    public void MakeCall(string number, int seconds)
    {
        MakeCall(number);
        Logger.Log($"Длительность звонка: {seconds} сек.");
    }

    public void UpgradeStorage(int extraGb)
    {
        StorageGb += extraGb;
        Logger.Log($"Память {Name} увеличена на {extraGb} ГБ. Теперь: {StorageGb} ГБ.");
    }

    public void SyncWith(SmartWatch watch)
    {
        IRechargeable rechargeable = watch;
        Logger.Log($"{Name} синхронизирован с часами «{watch.Name}» (заряд часов: {rechargeable.BatteryLevel}%).");
        if (rechargeable.BatteryLevel < 20)
            Logger.Log("Внимание: у часов низкий заряд!");
    }

    public double EstimateTalkTimeHours()
    {
        return BatteryMah / 300.0;
    }

    public void SetNfc(bool enabled)
    {
        HasNfc = enabled;
        Logger.Log($"{Name}: NFC {(enabled ? "включён" : "выключен")}.");
    }

    public void AddContact(string contactName, string number)
    {
        _contacts[contactName] = number;
        Logger.Log($"{Name}: контакт «{contactName}» сохранён.");
    }

    public bool InstallApp(string app)
    {
        if (!_installedApps.Add(app))
        {
            Logger.Log($"{Name}: приложение «{app}» уже установлено.");
            return false;
        }
        Logger.Log($"{Name}: установлено приложение «{app}».");
        AppInstalled?.Invoke(this, app);
        return true;
    }

    public bool CallContact(string contactName)
    {
        string number;
        if (!_contacts.TryGetValue(contactName, out number))
        {
            Logger.Log($"{Name}: контакт «{contactName}» не найден.");
            return false;
        }
        MakeCall(number);
        return true;
    }

    public List<string> GetRecentCalls(int count)
    {
        string[] all = _callHistory.ToArray();
        List<string> result = new List<string>();
        for (int i = all.Length - 1; i >= 0 && result.Count < count; i--)
            result.Add(all[i]);
        return result;
    }
}

public class GamingSmartphone : Smartphone
{
    private const int OverheatLimit = 45;

    private int _refreshRate;
    private bool _hasCooling;
    private string _gpu;
    private int _boostLevel;
    private int _temperature;
    private Dictionary<string, int> _playTime = new Dictionary<string, int>();
    private List<int> _boostHistory = new List<int>();

    public event Action<GamingSmartphone, int> Overheated;

    public int RefreshRate
    {
        get { return _refreshRate; }
        set { _refreshRate = value; }
    }

    public bool HasCooling
    {
        get { return _hasCooling; }
        set { _hasCooling = value; }
    }

    public string Gpu
    {
        get { return _gpu; }
        set { _gpu = value; }
    }

    public int BoostLevel
    {
        get { return _boostLevel; }
        private set { _boostLevel = value; }
    }

    public int Temperature
    {
        get { return _temperature; }
        private set { _temperature = value; }
    }

    public IReadOnlyDictionary<string, int> PlayTime
    {
        get { return _playTime; }
    }

    public IReadOnlyList<int> BoostHistory
    {
        get { return _boostHistory; }
    }

    public GamingSmartphone(string name, double price, string manufacturer, string article, int quantity, string country,
                            int warrantyPeriod, string model, int powerWatts, string color,
                            int storageGb, double screenInches, string operatingSystem,
                            int refreshRate, bool hasCooling, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, warrantyPeriod, model, powerWatts, color,
               storageGb, screenInches, operatingSystem, services)
    {
        RefreshRate = refreshRate;
        HasCooling = hasCooling;
        Gpu = "Adreno";
        BoostLevel = 0;
        Temperature = 30;
    }

    public override string GetCategory()
    {
        return "Электроника / Игровые смартфоны";
    }

    public override string GetTechInfo()
    {
        return base.GetTechInfo() + $", Частота экрана: {RefreshRate} Гц, Охлаждение: {(HasCooling ? "есть" : "нет")}, GPU: {Gpu}";
    }

    public void LaunchGame(string game)
    {
        Logger.Log($"{Name}: запуск игры «{game}» на {RefreshRate} Гц" + (HasCooling ? " (охлаждение включено)." : "."));
    }

    public void Boost(int level)
    {
        if (!HasCooling && level > 1)
        {
            Logger.Log($"{Name}: без охлаждения максимальный уровень буста — 1.");
            level = 1;
        }
        BoostLevel = level;
        _boostHistory.Add(level);
        Logger.Log($"{Name}: уровень буста {BoostLevel}.");
    }

    public bool IsHighRefreshRate()
    {
        return RefreshRate >= 120;
    }

    public void PlayGame(string game, int minutes)
    {
        if (minutes <= 0)
            throw new ArgumentOutOfRangeException(nameof(minutes), "Длительность должна быть больше нуля!");
        LaunchGame(game);
        int played;
        _playTime.TryGetValue(game, out played);
        _playTime[game] = played + minutes;
        Temperature += minutes / (HasCooling ? 5 : 2);
        Logger.Log($"{Name}: температура {Temperature}°C.");
        if (Temperature > OverheatLimit)
            Overheated?.Invoke(this, Temperature);
    }

    public void CoolDown(int degrees)
    {
        Temperature = Math.Max(30, Temperature - degrees);
        Logger.Log($"{Name}: остывание до {Temperature}°C.");
    }

    public int GetTotalPlayTime()
    {
        int total = 0;
        foreach (KeyValuePair<string, int> entry in _playTime)
            total += entry.Value;
        return total;
    }

    public string GetFavoriteGame()
    {
        string favorite = null;
        int max = 0;
        foreach (KeyValuePair<string, int> entry in _playTime)
        {
            if (entry.Value > max)
            {
                max = entry.Value;
                favorite = entry.Key;
            }
        }
        return favorite;
    }
}

public class SmartWatch : Electronics, IRechargeable, IWaterproof
{
    private int _batteryLevel;
    private int _waterResistanceMeters;
    private string _strapMaterial;
    private int _stepCount;
    private bool _hasGps;
    private List<int> _heartRates = new List<int>();
    private SortedSet<TimeSpan> _alarms = new SortedSet<TimeSpan>();
    private Dictionary<string, int> _workouts = new Dictionary<string, int>();

    public event Action<SmartWatch, int> LowBattery;

    public string StrapMaterial
    {
        get { return _strapMaterial; }
        set { _strapMaterial = value; }
    }

    public int StepCount
    {
        get { return _stepCount; }
        private set { _stepCount = value; }
    }

    public bool HasGps
    {
        get { return _hasGps; }
        set { _hasGps = value; }
    }

    public IReadOnlyList<int> HeartRates
    {
        get { return _heartRates; }
    }

    public IReadOnlyCollection<TimeSpan> Alarms
    {
        get { return _alarms; }
    }

    public IReadOnlyDictionary<string, int> Workouts
    {
        get { return _workouts; }
    }

    public SmartWatch(string name, double price, string manufacturer, string article, int quantity, string country,
                      int warrantyPeriod, string model, int powerWatts, string color,
                      int batteryLevel, int waterResistanceMeters, string strapMaterial, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, warrantyPeriod, model, powerWatts, color, services)
    {
        SetBattery(batteryLevel);
        _waterResistanceMeters = waterResistanceMeters;
        StrapMaterial = strapMaterial;
        StepCount = 0;
        HasGps = true;
    }

    private void SetBattery(int value)
    {
        if (value < 0 || value > 100)
            throw new ArgumentOutOfRangeException(nameof(value), "Заряд должен быть от 0 до 100%!");
        _batteryLevel = value;
        if (value < 20)
            LowBattery?.Invoke(this, value);
    }

    private bool CheckSwim()
    {
        return _waterResistanceMeters >= 50;
    }

    int IRechargeable.BatteryLevel
    {
        get { return _batteryLevel; }
        set { SetBattery(value); }
    }

    void IRechargeable.Charge(int percent)
    {
        SetBattery(Math.Min(100, _batteryLevel + percent));
        Logger.Log($"{Name}: зарядка +{percent}%. Текущий заряд: {_batteryLevel}%.");
    }

    string IRechargeable.GetStatus()
    {
        return $"Батарея: {_batteryLevel}%";
    }

    int IWaterproof.WaterResistanceMeters
    {
        get { return _waterResistanceMeters; }
        set { _waterResistanceMeters = value; }
    }

    bool IWaterproof.CanSwim()
    {
        return CheckSwim();
    }

    string IWaterproof.GetStatus()
    {
        return $"Водозащита: {_waterResistanceMeters} м ({(CheckSwim() ? "можно плавать" : "только брызги")})";
    }

    public override string GetCategory()
    {
        return "Электроника / Умные часы";
    }

    public override string GetTechInfo()
    {
        return base.GetTechInfo() + $", Заряд: {_batteryLevel}%, Водозащита: {_waterResistanceMeters} м, Ремешок: {StrapMaterial}";
    }

    public void AddSteps(int steps)
    {
        StepCount += steps;
        Logger.Log($"{Name}: шагов за день — {StepCount}.");
    }

    public double GetGoalProgress(int goal)
    {
        return Math.Min(100.0, StepCount * 100.0 / goal);
    }

    public void RecordHeartRate(int bpm)
    {
        if (bpm < 30 || bpm > 220)
            throw new ArgumentOutOfRangeException(nameof(bpm), "Пульс вне допустимого диапазона!");
        _heartRates.Add(bpm);
        Logger.Log($"{Name}: пульс {bpm} уд/мин.");
    }

    public bool AddAlarm(TimeSpan time)
    {
        bool added = _alarms.Add(time);
        string text = time.ToString("hh\\:mm");
        Logger.Log(added ? $"{Name}: будильник на {text}." : $"{Name}: будильник на {text} уже есть.");
        return added;
    }

    public void StartWorkout(string type, int minutes)
    {
        if (minutes <= 0)
            throw new ArgumentOutOfRangeException(nameof(minutes), "Длительность должна быть больше нуля!");
        int done;
        _workouts.TryGetValue(type, out done);
        _workouts[type] = done + minutes;
        Logger.Log($"{Name}: тренировка «{type}», {minutes} мин.");
        SetBattery(Math.Max(0, _batteryLevel - minutes / 4));
    }

    public double GetAverageHeartRate()
    {
        if (_heartRates.Count == 0)
            return 0;
        int sum = 0;
        foreach (int bpm in _heartRates)
            sum += bpm;
        return (double)sum / _heartRates.Count;
    }
}

public class AudioBook : Books, IPlayable
{
    private int _durationMinutes;
    private string _narrator;
    private double _playbackSpeed;
    private int _currentMinute;
    private SortedDictionary<int, string> _chapterMarks = new SortedDictionary<int, string>();
    private Stack<int> _positionHistory = new Stack<int>();
    private bool _isPlaying;

    public event Action<AudioBook, bool> PlaybackChanged;

    public string Narrator
    {
        get { return _narrator; }
        set { _narrator = value; }
    }

    public double PlaybackSpeed
    {
        get { return _playbackSpeed; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Скорость должна быть положительной!");
            _playbackSpeed = value;
        }
    }

    public int CurrentMinute
    {
        get { return _currentMinute; }
        private set { _currentMinute = value; }
    }

    public bool IsPlaying
    {
        get { return _isPlaying; }
        private set { _isPlaying = value; }
    }

    public IReadOnlyDictionary<int, string> ChapterMarks
    {
        get { return _chapterMarks; }
    }

    public int HistoryDepth
    {
        get { return _positionHistory.Count; }
    }

    public AudioBook(string name, double price, string manufacturer, string article, int quantity, string country,
                     string author, string genre, int pages, int year,
                     int durationMinutes, string narrator, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, author, genre, pages, year, services)
    {
        _durationMinutes = durationMinutes;
        Narrator = narrator;
        PlaybackSpeed = 1.0;
        CurrentMinute = 0;
        IsPlaying = false;
    }

    int IPlayable.DurationMinutes
    {
        get { return _durationMinutes; }
        set { _durationMinutes = value; }
    }

    void IPlayable.Play()
    {
        IsPlaying = true;
        Logger.Log($"Воспроизведение «{Name}» (читает {Narrator}), длительность {_durationMinutes / 60} ч {_durationMinutes % 60} мин, скорость x{PlaybackSpeed}.");
        PlaybackChanged?.Invoke(this, true);
    }

    void IPlayable.Pause()
    {
        IsPlaying = false;
        Logger.Log($"«{Name}»: пауза на {CurrentMinute}-й минуте.");
        PlaybackChanged?.Invoke(this, false);
    }

    public override string GetCategory()
    {
        return "Книги / Аудиокниги";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Чтец: {Narrator}";
    }

    public void Rewind(int minutes)
    {
        CurrentMinute = Math.Max(0, Math.Min(_durationMinutes, CurrentMinute + minutes));
        Logger.Log($"«{Name}»: перемотка, текущая минута — {CurrentMinute}.");
    }

    public double GetProgress()
    {
        return CurrentMinute * 100.0 / _durationMinutes;
    }

    public void AddChapterMark(string title, int minute)
    {
        if (minute < 0 || minute > _durationMinutes)
            throw new ArgumentOutOfRangeException(nameof(minute), "Минута вне длительности аудиокниги!");
        _chapterMarks[minute] = title;
        Logger.Log($"«{Name}»: метка «{title}» на {minute}-й минуте.");
    }

    public bool JumpToChapter(string title)
    {
        foreach (KeyValuePair<int, string> mark in _chapterMarks)
        {
            if (mark.Value == title)
            {
                _positionHistory.Push(CurrentMinute);
                CurrentMinute = mark.Key;
                Logger.Log($"«{Name}»: переход к «{title}» ({CurrentMinute}-я минута).");
                return true;
            }
        }
        Logger.Log($"«{Name}»: глава «{title}» не найдена.");
        return false;
    }

    public bool GoBack()
    {
        if (_positionHistory.Count == 0)
            return false;
        CurrentMinute = _positionHistory.Pop();
        Logger.Log($"«{Name}»: возврат на {CurrentMinute}-ю минуту.");
        return true;
    }

    public string GetCurrentChapter()
    {
        string current = "Без главы";
        foreach (KeyValuePair<int, string> mark in _chapterMarks)
        {
            if (mark.Key <= CurrentMinute)
                current = mark.Value;
        }
        return current;
    }
}

public class Inventory<T> where T : Product
{
    private string _title;
    private List<T> _items = new List<T>();
    private Dictionary<string, T> _byArticle = new Dictionary<string, T>();
    private ILogger _logger;

    public event Action<T> ItemAdded;
    public event Action<T> ItemRemoved;

    public string Title
    {
        get { return _title; }
        set { _title = value; }
    }

    public int Count
    {
        get { return _items.Count; }
    }

    public Inventory(string title, ILogger logger)
    {
        Title = title;
        _logger = logger ?? throw new ArgumentNullException(nameof(logger));
    }

    public void Add(T item)
    {
        _items.Add(item);
        _byArticle[item.Article] = item;
        _logger.Log($"[{Title}] добавлено: {item.Name}");
        ItemAdded?.Invoke(item);
    }

    public bool Remove(T item)
    {
        bool removed = _items.Remove(item);
        if (removed)
        {
            _byArticle.Remove(item.Article);
            _logger.Log($"[{Title}] удалено: {item.Name}");
            ItemRemoved?.Invoke(item);
        }
        return removed;
    }

    public T FindByName(string name)
    {
        return _items.Find(p => p.Name == name);
    }

    public T FindByArticle(string article)
    {
        T result;
        return _byArticle.TryGetValue(article, out result) ? result : null;
    }

    public T GetMostExpensive()
    {
        T result = null;
        foreach (T item in _items)
        {
            if (result == null || item.Price > result.Price)
                result = item;
        }
        return result;
    }

    public double GetTotalCost()
    {
        double sum = 0;
        foreach (T item in _items)
            sum += item.GetTotalCost();
        return sum;
    }

    public List<T> Filter(Func<T, bool> predicate)
    {
        return _items.FindAll(p => predicate(p));
    }

    public void Sort(Comparison<T> comparison)
    {
        _items.Sort(comparison);
    }

    public void ForEach(Action<T> action)
    {
        _items.ForEach(action);
    }

    public Dictionary<string, List<T>> GroupByCategory()
    {
        Dictionary<string, List<T>> groups = new Dictionary<string, List<T>>();
        foreach (T item in _items)
        {
            string category = item.GetCategory();
            if (!groups.ContainsKey(category))
                groups[category] = new List<T>();
            groups[category].Add(item);
        }
        return groups;
    }

    public void ApplyDiscountToAll(double percent)
    {
        foreach (T item in _items)
            item.Discount(percent);
    }

    public void ShowAll()
    {
        Console.WriteLine($"=== {Title} (позиций: {Count}) ===");
        foreach (T item in _items)
        {
            Console.WriteLine($"[{item.GetCategory()}]");
            item.Display();
        }
    }
}

public class Bundle<TFirst, TSecond> where TFirst : Product where TSecond : Product
{
    private double _discountPercent;

    public TFirst First { get; set; }
    public TSecond Second { get; set; }

    public double DiscountPercent
    {
        get { return _discountPercent; }
        set { _discountPercent = value; }
    }

    public Bundle(TFirst first, TSecond second, double discountPercent)
    {
        First = first;
        Second = second;
        DiscountPercent = discountPercent;
    }

    public double GetPrice()
    {
        return (First.Price + Second.Price) * (1 - DiscountPercent / 100);
    }

    public void Show()
    {
        Console.WriteLine($"Комплект: «{First.Name}» + «{Second.Name}», скидка {DiscountPercent}%, цена: {GetPrice()} руб.");
    }
}

public static class ProductUtils
{
    public static T GetCheaper<T>(T first, T second) where T : Product
    {
        return first.Price <= second.Price ? first : second;
    }
}

ServiceContainer container = new ServiceContainer();
container.RegisterSingleton<ILogger>(new PrefixLogger("[LOG]"));
container.RegisterSingleton<IDiscountPolicy>(new LimitedDiscountPolicy(30));
container.RegisterSingleton<INotifier>(new ConsoleNotifier());
container.RegisterSingleton<StockMonitor>(new StockMonitor(container.Resolve<INotifier>(), (oldPrice, newPrice) => newPrice <= oldPrice * 0.75));
container.Register<IProductServices>(c => new ProductServices(c.Resolve<ILogger>(), c.Resolve<IDiscountPolicy>()));
container.Register<OrderService>(c => new OrderService(c.Resolve<ILogger>(), c.Resolve<INotifier>()));

IProductServices services = container.Resolve<IProductServices>();

Electronics tv = new Electronics("Телевизор", 40000, "LG", "EL-001", 5, "Корея", 24, "OLED55", 120, "Чёрный", services);
Clothing tshirt = new Clothing("Футболка", 1500, "Nike", "CL-010", 20, "Вьетнам", "L", "хлопок", "Лето", "Белый", services);
Books book = new Books("Война и мир", 800, "Эксмо", "BK-100", 15, "Россия", "Л.Н. Толстой", "Роман", 1225, 1869, services);
Smartphone phone = new Smartphone("Смартфон", 50000, "Samsung", "EL-200", 8, "Корея", 24, "Galaxy S", 15, "Серый", 256, 6.5, "Android", services);
GamingSmartphone gamePhone = new GamingSmartphone("Игровой смартфон", 80000, "ASUS", "EL-300", 3, "Тайвань", 12, "ROG Phone", 30, "Чёрный", 512, 6.8, "Android", 165, true, services);
SmartWatch watch = new SmartWatch("Умные часы", 15000, "Apple", "EL-400", 12, "Китай", 12, "Watch 9", 2, "Серебристый", 15, 50, "Силикон", services);

IProductServices quietServices = new ProductServices(new SilentLogger(), new StandardDiscountPolicy());
AudioBook audio = new AudioBook("Мастер и Маргарита (аудио)", 600, "Строки", "BK-200", 100, "Россия", "М.А. Булгаков", "Роман", 480, 1967, 1230, "Сергей Чонишвили", quietServices);

StockMonitor monitor = container.Resolve<StockMonitor>();
foreach (Product watched in new Product[] { tv, tshirt, gamePhone, watch })
    monitor.Watch(watched);

Console.WriteLine("===== Полиморфизм: перекрытие методов =====");
List<Product> products = new List<Product> { tv, tshirt, book, phone, gamePhone, watch, audio };
foreach (Product p in products)
{
    Console.WriteLine($"[{p.GetCategory()}]");
    p.Display();
    Console.WriteLine();
}

Console.WriteLine("===== Полиморфизм: перегрузка методов =====");
tv.Sell(2);
tv.Sell(1, 10);
tv.AddToStock(5);
tv.AddToStock(10, "ООО Техноснаб");
tv.Discount(10);
tshirt.Discount(20, 200);
tv.ComparePriceWith(tshirt);
tv.ComparePriceWith(30000);
phone.MakeCall("+7 900 123-45-67");
phone.MakeCall("+7 900 123-45-67", 45);

Console.WriteLine("\n===== Явная реализация интерфейсов =====");
IRechargeable rechargeable = watch;
IWaterproof waterproof = watch;
rechargeable.Charge(50);
Console.WriteLine(rechargeable.GetStatus());
Console.WriteLine(waterproof.GetStatus());
Console.WriteLine($"Можно плавать: {waterproof.CanSwim()}");

IPlayable player = audio;
player.Play();
player.Pause();
audio.Rewind(30);
Console.WriteLine($"Прогресс аудиокниги (тихий логгер, но Console работает): {audio.GetProgress():F1}%");

gamePhone.LaunchGame("Genshin Impact");
gamePhone.Boost(3);
phone.SyncWith(watch);

Console.WriteLine("\n===== Управление зависимостями =====");
OrderService orders = container.Resolve<OrderService>();
orders.PlaceOrder(gamePhone, 2);
orders.PlaceOrder(tv, 1);

tv.Rate(4.5);
Console.WriteLine($"Цена телевизора с НДС: {tv.GetPriceWithVat()} руб.");
tshirt.PutOnSale(50);
watch.AddSteps(7500);
Console.WriteLine($"Прогресс цели по шагам: {watch.GetGoalProgress(10000)}%");

Console.WriteLine("\n===== Generic классы и коллекции =====");
ILogger logger = container.Resolve<ILogger>();
Inventory<Product> store = new Inventory<Product>("Главный склад", logger);
int addedCount = 0;
store.ItemAdded += added => addedCount++;
foreach (Product p in products)
    store.Add(p);
Console.WriteLine($"Событие ItemAdded сработало раз: {addedCount}");

Inventory<Electronics> techStore = new Inventory<Electronics>("Склад электроники", new SilentLogger());
techStore.Add(tv);
techStore.Add(phone);
techStore.Add(gamePhone);
techStore.Add(watch);

Console.WriteLine($"Самый дорогой в электронике: {techStore.GetMostExpensive().Name}");
Console.WriteLine($"Стоимость склада электроники: {techStore.GetTotalCost()} руб.");
techStore.ApplyDiscountToAll(10);
Console.WriteLine($"Стоимость после скидки: {techStore.GetTotalCost()} руб.");

foreach (Product cheap in store.Filter(p => p.Price < 2000))
    Console.WriteLine($"Дешёвый товар: {cheap.Name} — {cheap.Price} руб.");

Product found = store.FindByName("Футболка");
Console.WriteLine($"Поиск: {found.GetInfo()}");
Console.WriteLine($"Поиск по артикулу EL-200: {store.FindByArticle("EL-200").Name}");

store.Sort((first, second) => first.Price.CompareTo(second.Price));
Console.WriteLine("Товары по возрастанию цены:");
store.ForEach(x => Console.WriteLine($"  {x.Name}: {x.Price} руб."));

foreach (KeyValuePair<string, List<Product>> entry in store.GroupByCategory())
    Console.WriteLine($"Категория «{entry.Key}»: {entry.Value.Count} шт.");

Bundle<Smartphone, SmartWatch> bundle = new Bundle<Smartphone, SmartWatch>(phone, watch, 10);
bundle.Show();

Product cheaper = ProductUtils.GetCheaper<Product>(tv, tshirt);
Console.WriteLine($"Дешевле: {cheaper.Name}");
Electronics cheaperTech = ProductUtils.GetCheaper<Electronics>(phone, watch);
Console.WriteLine($"Дешевле из электроники: {cheaperTech.Name}");

Console.WriteLine("\n===== События и делегаты в базовом классе Product =====");
tv.StockChanged += (product, oldQty, newQty) => Console.WriteLine($"[СОБЫТИЕ] «{product.Name}»: остаток {oldQty} -> {newQty} шт.");
tv.PriceChanged += (sender, args) => Console.WriteLine($"[СОБЫТИЕ] Цена изменена: {args.OldPrice} -> {args.NewPrice} руб.");
tv.AddToStock(3);
tv.ApplyPriceModifier(price => price * 0.5);
tv.AddTag("Акция");
tv.AddTag("акция");
tv.AddTag("OLED");
Console.WriteLine($"Тег «акция» есть: {tv.HasTag("акция")}, всего тегов: {tv.Tags.Count}");
tv.AddReview("Иван", "Отличная картинка");
tv.AddReview("Иван", "Картинка отличная, звук средний");
Console.WriteLine($"Отзывов: {tv.Reviews.Count}, продано: {tv.SoldCount} шт.");
Console.WriteLine($"История цен: {string.Join(" -> ", tv.PriceHistory)}");
Console.WriteLine($"Минимальная цена за всё время: {tv.GetLowestPrice()} руб.");
gamePhone.Sell(1);
Console.WriteLine($"Оповещений монитора склада: {monitor.Alerts.Count}");

orders.OrderPlaced += (orderedProduct, orderedCount) => Console.WriteLine($"[СОБЫТИЕ] Заказ принят: «{orderedProduct.Name}» x{orderedCount}");
orders.PlaceOrder(phone, 4);
foreach (string order in orders.OrderHistory)
    Console.WriteLine($"История заказов: {order}");

Console.WriteLine("\n===== Electronics =====");
tv.PowerStateChanged += (device, isOn) => Console.WriteLine($"[СОБЫТИЕ] {device.Name}: {(isOn ? "включён" : "выключен")}");
tv.TogglePower();
tv.TogglePower();
tv.AddAccessory("Пульт ДУ");
tv.AddAccessory("Кронштейн");
tv.AddAccessory("Подставка");
tv.RegisterRepair("Замена блока питания");
foreach (string accessory in tv.FindAccessories(a => a.StartsWith("П")))
    Console.WriteLine($"Аксессуар на «П»: {accessory}");
Console.WriteLine($"Записей о ремонте: {tv.RepairLog.Count}");

Console.WriteLine("\n===== Clothing =====");
tshirt.SizeSoldOut += (cloth, size) => Console.WriteLine($"[СОБЫТИЕ] Размер {size} для «{cloth.Name}» распродан");
tshirt.AddSizeStock("M", 5);
tshirt.AddSizeStock("L", 3);
tshirt.SellSize("L", 3);
tshirt.SellSize("M", 10);
tshirt.AddCareTag("не отбеливать");
tshirt.AddCareTag("не отбеливать");
tshirt.RecordWash();
Console.WriteLine($"Размеров на складе: {tshirt.SizeStock.Count}, тегов ухода: {tshirt.CareTags.Count}");

Console.WriteLine("\n===== Books =====");
book.BookFinished += finished => Console.WriteLine($"[СОБЫТИЕ] Книга «{finished.Name}» прочитана");
book.AddChapter("Том первый");
book.AddBookmark("Бал Наташи", 310);
book.ReadTo(600);
book.ReadTo(1225);
Console.WriteLine($"Прогресс чтения: {book.GetReadProgress():F1}%");

Console.WriteLine("\n===== Smartphone =====");
phone.AppInstalled += (owner, app) => Console.WriteLine($"[СОБЫТИЕ] {owner.Name}: установлено «{app}»");
phone.AddContact("Мама", "+7 900 111-22-33");
phone.InstallApp("Telegram");
phone.InstallApp("Telegram");
phone.CallContact("Мама");
phone.CallContact("Папа");
foreach (string call in phone.GetRecentCalls(3))
    Console.WriteLine($"Недавний звонок: {call}");
Console.WriteLine($"Приложений: {phone.InstalledApps.Count}, звонков в истории: {phone.CallCount}");

Console.WriteLine("\n===== GamingSmartphone =====");
gamePhone.Overheated += (device, temp) => Console.WriteLine($"[СОБЫТИЕ] Перегрев {device.Name}: {temp}°C");
gamePhone.PlayGame("Genshin Impact", 60);
gamePhone.PlayGame("Call of Duty", 90);
gamePhone.CoolDown(10);
Console.WriteLine($"Любимая игра: {gamePhone.GetFavoriteGame()}, всего минут: {gamePhone.GetTotalPlayTime()}");
Console.WriteLine($"Бустов в истории: {gamePhone.BoostHistory.Count}");

Console.WriteLine("\n===== SmartWatch =====");
watch.LowBattery += (device, level) => Console.WriteLine($"[СОБЫТИЕ] Низкий заряд {device.Name}: {level}%");
watch.RecordHeartRate(72);
watch.RecordHeartRate(85);
watch.AddAlarm(new TimeSpan(7, 30, 0));
watch.AddAlarm(new TimeSpan(7, 30, 0));
watch.StartWorkout("Бег", 200);
Console.WriteLine($"Средний пульс: {watch.GetAverageHeartRate()}, будильников: {watch.Alarms.Count}, тренировок: {watch.Workouts.Count}");

Console.WriteLine("\n===== AudioBook =====");
audio.PlaybackChanged += (track, playing) => Console.WriteLine($"[СОБЫТИЕ] «{track.Name}»: {(playing ? "воспроизведение" : "пауза")}");
audio.AddChapterMark("Глава 1", 0);
audio.AddChapterMark("Глава 2", 45);
audio.AddChapterMark("Глава 3", 120);
audio.JumpToChapter("Глава 3");
Console.WriteLine($"Текущая глава: {audio.GetCurrentChapter()}");
audio.GoBack();
Console.WriteLine($"После возврата: {audio.GetCurrentChapter()}, минута {audio.CurrentMinute}");
player.Play();
player.Pause();

===== Полиморфизм: перекрытие методов =====
[Электроника]
Название: Телевизор, Цена: 40000 руб., Производитель: LG
Модель: OLED55, Мощность: 120 Вт, Цвет: Чёрный, Гарантия: 24 мес.

[Одежда]
Название: Футболка, Цена: 1500 руб., Производитель: Nike, Размер: L, Материал: хлопок

[Книги]
Название: Война и мир, Цена: 800 руб., Производитель: Эксмо, Автор: Л.Н. Толстой, Жанр: Роман

[Электроника / Смартфоны]
Название: Смартфон, Цена: 50000 руб., Производитель: Samsung
Модель: Galaxy S, Мощность: 15 Вт, Цвет: Серый, Гарантия: 24 мес., Память: 256 ГБ, Экран: 6,5", ОС: Android

[Электроника / Игровые смартфоны]
Название: Игровой смартфон, Цена: 80000 руб., Производитель: ASUS
Модель: ROG Phone, Мощность: 30 Вт, Цвет: Чёрный, Гарантия: 12 мес., Память: 512 ГБ, Экран: 6,8", ОС: Android, Частота экрана: 165 Гц, Охлаждение: есть, GPU: Adreno

[Электроника / Умные часы]
Название: Умные часы, Цена: 15000 руб., Производитель: Apple
Модель: Watch 9, Мощность: 2 Вт, Цвет: Серебристый, Гарантия: 12 мес.